# Практика 31 · Як модель породжує текст

> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

Зошит самодостатній: усе, що тут відбувається, пояснюється на місці, і лекцію
відкривати не обовʼязково.

**Задача, яку ми розвʼязуємо.** У нас є мовна модель, яка вміє одне: сказати,
наскільки ймовірне кожне наступне слово. Нам треба з цього зробити ціле
продовження речення. Способів зробити це кілька, вони дають дуже різний текст,
і ми **заміряємо**, який чим платить.

Що зробимо:

1. зберемо корпус із українських перекладів, що лежать у системі, і навчимо
   маленьку мовну модель;
2. подивимось, який вигляд має розподіл наступного слова — і скільки в ньому
   кандидатів насправді;
3. напишемо **своїми руками** пʼять способів декодування: жадібне, семплювання
   з температурою, `top-k`, `top-p` і `beam search` з нормалізацією на довжину;
4. заміряємо кожен пʼятьма величинами й побачимо, що правдоподібність і якість
   тягнуть у різні боки;
5. увімкнемо примусову довжину — і побачимо **виродження**: цикл, який модель
   не може розірвати;
6. додамо два лікування — **штраф за повтор** і **заборону повторених
   n-грам** — і подивимось, що кожне дає, а чого наш стенд про них показати не
   може.

> ⏱ **Зошит навчає одну мовну модель і проганяє близько тридцяти прогонів
> декодування.** Заміряно: **близько восьми хвилин процесорного часу** (у нашому прогоні 481 секунда) на чотирьох ядрах без відеокарти,
> **в один потік**. За годинником вийде помітно більше, якщо машина зайнята —
> саме тому ми міряємо `time.process_time()`, а не годинник. Власний
> процесорний час зошит друкує останньою клітинкою: звір із цим числом.

> ⚠️ **Твої числа не збіжаться з тими, що в лекції, і це нормально.** Корпус ми
> беремо з файлів локалізації, встановлених **на твоїй машині**, а в кожного
> свій набір програм. Відтворюється тут не значення, а **напрямок**: жадібне
> декодування завжди дає найкоротший вихід і найвищий логарифм імовірності,
> семплювання завжди різноманітніше, а ширший промінь завжди вкорочує
> відповідь.

## 0 · Середовище й чому ми фіксуємо потоки

Зошит міряє час, а час на спільній машині бреше двома способами.

**Стінний годинник** показує, скільки минуло реального часу. Якщо поряд
рахується щось іще, він покаже більше, і число нічого не варте.

**Процесорний годинник** (`time.process_time()`) рахує лише той час, коли
процесор працював саме над нашою програмою. Але й у нього є пастка: якщо
бібліотека розкладає роботу на потоки, то потоки, які **чекають**, теж
зараховуються як робота. Тому спершу фіксуємо один потік — і робимо це **до**
імпорту `torch`, бо змінні середовища читаються під час завантаження
бібліотеки.

Заразом це робить зошит **відтворюваним**: в один потік порядок додавання
чисел завжди той самий.

In [ ]:
import os
# ⚠️ ці три рядки мусять стояти ДО імпорту torch — інакше бібліотеки вже
# запустять свої потоки, і процесорний час стане брехливим
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, glob, gettext, re, math, time, random, collections, statistics
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.set_num_threads(1)

START_CPU = time.process_time()          # звідси рахуємо власний час зошита

print('python ', sys.version.split()[0])
print('torch  ', torch.__version__)
print('ядер   ', os.cpu_count())
print('навантаження машини', round(os.getloadavg()[0], 2))

## 1 · Корпус: те, що вже лежить у системі

Мережа в зошиті заборонена, і вона нам не потрібна. У кожній системі з
українською локаллю лежить каталог `/usr/share/locale/uk/LC_MESSAGES/` — це
скомпільовані переклади інтерфейсів програм. Кожен файл `.mo` — словник
«англійський оригінал → український переклад». Переклад писала людина, тобто це
справжній текст, а не згенерований.

Читати `.mo` вміє стандартний модуль `gettext`. Ми беремо **тільки український
бік** — нам потрібна одномовна мовна модель.

Перш ніж щось рахувати, перевіримо, що дані взагалі є. Якщо української локалі
в системі немає, зошит має сказати це людською мовою, а не впасти стеком на
двадцятій клітинці.

In [ ]:
LOCALE_DIR = '/usr/share/locale/uk/LC_MESSAGES'

# канонічний для курсу словесний вираз: апостроф усередині слова — звʼязка,
# а не межа, тому «зʼєднання» лишається одним словом
WORD = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")


def load_messages(min_chars=20):
    """Повертає список списків слів: по одному рядку перекладу на елемент."""
    out = []
    for path in sorted(glob.glob(LOCALE_DIR + '/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue                      # зламаний каталог просто пропускаємо
        for source, target in catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if (isinstance(source, str) and isinstance(target, str)
                    and len(target) > min_chars and 'Project-Id' not in target):
                out.append(WORD.findall(target.lower()))
    return out


if not os.path.isdir(LOCALE_DIR):
    raise SystemExit(
        'У цій системі немає теки ' + LOCALE_DIR + '.\n'
        'Зошитові немає на чому працювати. Постав українську локаль\n'
        '(наприклад, пакет langpacks-uk у Fedora чи language-pack-uk в Ubuntu)\n'
        'або підстав власний корпус українських речень у load_messages().')

lines = load_messages()
if len(lines) < 5000:
    raise SystemExit(
        'Знайдено лише ' + str(len(lines)) + ' рядків перекладу — цього замало,\n'
        'щоб навчити мовну модель. Постав більше українських локалей\n'
        'або підстав власний корпус.')

print('рядків перекладу      ', len(lines))
print('слововживань          ', sum(len(s) for s in lines))
print('різних словоформ      ', len({w for s in lines for w in s}))
print()
print('приклад:', ' '.join(lines[0][:12]))

## 2 · Словник і поділ на три частини

Модель працює не зі словами, а з **токенами** — числами. Тому спершу будуємо
словник: усі слова, що трапились у навчальній частині щонайменше пʼять разів.
Рідкісні слова заміняємо службовим токеном `<unk>`: інакше словник розпухне
хвостом, який модель усе одно не вивчить.

Крім слів, у словнику є чотири службові токени:

| токен | навіщо |
|---|---|
| `<pad>` | добивання коротких рядків до спільної довжини в пачці |
| `<bos>` | початок рядка; з нього модель починає породжувати |
| `<eos>` | кінець рядка; **саме його обирає модель, коли хоче зупинитись** |
| `<unk>` | будь-яке слово, якого немає в словнику |

Ділимо корпус на три частини: **навчальну** (на ній учиться модель),
**відкладену** (на ній добирали швидкість навчання) і **перевірну** (на ній
оголошуємо результат, модель її не бачить ніколи).

⚠️ Перед поділом **перемішуємо** з фіксованим зерном. Рядки лежать у порядку
програм: спершу всі рядки `abrt`, потім `accountsservice` і так далі. Якби ми
взяли «перші десять відсотків», це була б не менша вибірка, а **вужчий
домен** — пів десятка програм замість двохсот шістдесяти.

In [ ]:
MAXLEN = 32                    # найдовший рядок, з яким працює модель
PAD, BOS, EOS, UNK = 0, 1, 2, 3

rows = [s for s in lines if 4 <= len(s) <= MAXLEN - 2]
random.Random(0).shuffle(rows)              # обовʼязково: у даних є порядок

n = len(rows)
train_rows, hold_rows, test_rows = rows[:int(0.8*n)], rows[int(0.8*n):int(0.9*n)], rows[int(0.9*n):]

counts = collections.Counter(w for s in train_rows for w in s)
itos = ['<pad>', '<bos>', '<eos>', '<unk>'] + [w for w, c in counts.most_common() if c >= 5]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)


def encode(words):
    """Слова -> числа, з початком і кінцем рядка."""
    return [BOS] + [stoi.get(w, UNK) for w in words] + [EOS]


train = [encode(s) for s in train_rows]
hold = [encode(s) for s in hold_rows]
test = [encode(s) for s in test_rows]

print('словник        ', V)
print('навчальних     ', len(train))
print('відкладених    ', len(hold))
print('перевірних     ', len(test))
print('середня довжина', round(sum(len(s) for s in train) / len(train), 2), 'токенів')

## 3 · Модель

Беремо найпростіший трансформер-декодер: ембединги слів, ембединги позицій, два
шари з self-attention і причинною маскою, лінійний вихід на весь словник.
Причинна маска — це заборона дивитись уперед: токен на позиції 5 бачить позиції
1-5 і не бачить 6, 7… Без неї модель просто підглянула б відповідь.

Це не найкраща архітектура, а найдешевша, у якій видно все, що нам потрібно.

In [ ]:
class SmallLM(nn.Module):
    def __init__(self, vocab, d=128, layers=2, heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = nn.Embedding(MAXLEN, d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4*d,
                                           batch_first=True, dropout=0.0, norm_first=True)
        self.body = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        length = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(length))
        # причинна маска: −нескінченність вище головної діагоналі
        mask = torch.triu(torch.full((length, length), float('-inf')), 1)
        return self.head(self.body(h, mask=mask, src_key_padding_mask=(x == PAD)))


def pad_batch(rows):
    """Добиває рядки пачки до спільної довжини токеном <pad>."""
    width = max(len(r) for r in rows)
    return torch.tensor([r + [PAD] * (width - len(r)) for r in rows])


model_probe = SmallLM(V)
print('ваг у моделі', sum(p.numel() for p in model_probe.parameters()))
del model_probe

## 4 · Швидкість навчання: чому саме така

Швидкість навчання — головна ручка. Щоб її не вгадувати, ми прогнали
логарифмічну сітку з пʼятьох точок **на тому самому бюджеті**, на якому
навчатимемо (тисяча кроків), і дивились на перплексію **відкладеної** частини —
не перевірної, інакше це було б підглядання у відповідь.

Наші числа (на нашій машині, з нашим набором локалей):

| швидкість навчання | 0.001 | 0.003 | 0.01 | 0.02 | 0.03 |
|---|---|---|---|---|---|
| перплексія відкладеної | 201.51 | 150.14 | 139.10 | 135.21 | 136.91 |

Мінімум усередині сітки, тож беремо **0.02**. Сам добір тут не
повторюємо — це пʼять навчань замість одного, і зошит став би вп'ятеро довшим.
Скрипт добору лежить поруч із рештою замірів теми; якщо хочеш перевірити,
поміняй `LR` нижче й подивись, як зміниться перплексія.

**Перплексія** — це міра того, наскільки текст дивує модель: приблизно між
скількома варіантами вона вагається на кожному кроці. Менше — краще. Поруч
рахуємо **уніграмний рубіж**: перплексію найдурнішої моделі, яка не дивиться на
контекст узагалі, а просто називає слова за їхньою частотою. Якщо наша модель
не нижча за нього — вона нічого не вивчила.

In [ ]:
LR = 0.02          # дібрано сіткою на відкладеній частині, див. таблицю вище
STEPS = 1000            # бюджет навчання
BATCH = 64
SEED = 0


def train_model(steps=STEPS, lr=LR, seed=SEED):
    torch.manual_seed(seed)
    net = SmallLM(V)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps,
                                                pct_start=0.1)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    rng = random.Random(seed)
    index = list(range(len(train)))
    t0 = time.process_time()
    for _ in range(steps):
        batch = pad_batch([train[j] for j in rng.sample(index, BATCH)])
        # вхід — усе, крім останнього токена; ціль — усе, крім першого:
        # так одне речення дає стільки задач «вгадай наступне», скільки в ньому слів
        loss = loss_fn(net(batch[:, :-1]).reshape(-1, V), batch[:, 1:].reshape(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
    return net, time.process_time() - t0


model, train_sec = train_model()
model.eval()
print('навчено за', round(train_sec, 1), 'с процесорного часу')

In [ ]:
@torch.no_grad()
def perplexity(net, data, batch=128):
    """Перплексія: e у степені середньої втрати на токен."""
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
    total, count = 0.0, 0
    for i in range(0, len(data), batch):
        x = pad_batch(data[i:i + batch])
        total += loss_fn(net(x[:, :-1]).reshape(-1, V), x[:, 1:].reshape(-1)).item()
        count += int((x[:, 1:] != PAD).sum())
    return math.exp(total / count)


# уніграмний рубіж: модель, яка не дивиться на контекст
uni_counts = collections.Counter(t for s in train for t in s[1:])
uni_total = sum(uni_counts.values())
uni_nats = -sum(c * math.log(c / uni_total) for c in uni_counts.values()) / uni_total

PPL_HOLD = perplexity(model, hold)
PPL_TEST = perplexity(model, test)
print('перплексія відкладеної', round(PPL_HOLD, 2))
print('перплексія перевірної ', round(PPL_TEST, 2))
print('уніграмний рубіж      ', round(math.exp(uni_nats), 2))
print()
print('модель нижча за рубіж у', round(math.exp(uni_nats) / PPL_TEST, 2), 'раза —',
      'отже контекст вона таки читає' if PPL_TEST < math.exp(uni_nats) else 'ЩОСЬ НЕ ТАК')

## 5 · Задача, на якій порівнюватимемо

Однакова для всіх способів декодування: **дано перші три слова перевірного
рядка — допиши решту**. Ми знаємо, що там стояло насправді, тож можемо
порівняти.

Беремо рядки, у яких після трьох слів лишається ще щонайменше пʼять — інакше
дописувати нема чого.

In [ ]:
PREFIX = 3             # скільки слів підказки
MAXNEW = 20            # найдовше продовження, яке дозволяємо
NPROMPT = 150          # скільки підказок у замірі
NBEAM = 100            # промінь рахуємо на меншій вибірці: він найдорожчий

long_rows = [r for r in test if len(r) >= PREFIX + 6][:NPROMPT]
prompts = [r[:PREFIX + 1] for r in long_rows]                    # <bos> + три слова
refs = [[t for t in r[PREFIX + 1:] if t != EOS] for r in long_rows]

def show(tokens):
    return ' '.join(itos[t] for t in tokens)

print('підказок                ', len(prompts))
print('середня довжина еталона ', round(statistics.mean(len(r) for r in refs), 2), 'токенів')
print()
for i in range(3):
    print('  підказка:', show(prompts[i][1:]))
    print('  далі йшло:', show(refs[i]))
    print()

## 6 · Який вигляд має розподіл наступного слова

Перш ніж щось вибирати, подивімось, з чого вибираємо. Візьмемо кожну підказку,
попросимо модель дати розподіл наступного слова й порахуємо три речі.

**Ентропія** — міра невизначеності в бітах: скільки двійкових питань «так чи ні»
треба в середньому, щоб угадати відповідь. Нуль означає «модель певна»; верхня
межа для словника з `V` слів — `log₂ V`.

**Скільки слів набирає задану масу** — впорядкуємо слова за спаданням
імовірності й порахуємо, скільки треба скласти, щоб їхня сума вперше перевищила
поріг. Саме це число й регулює `top-p`.

**Скільки маси забирають k найкращих** — те саме з іншого боку, і саме це
регулює `top-k`.

In [ ]:
@torch.no_grad()
def next_word_probs(seqs):
    """Розподіл наступного слова для пачки послідовностей."""
    x = pad_batch(seqs)
    logits = model(x)
    # для кожного рядка беремо позицію його ОСТАННЬОГО справжнього токена
    last = torch.stack([logits[i, len(seqs[i]) - 1] for i in range(len(seqs))])
    return F.softmax(last, dim=-1)


P1 = next_word_probs(prompts)
entropy_bits = -(P1 * torch.log2(P1.clamp_min(1e-12))).sum(-1)
sorted_p, _ = P1.sort(-1, descending=True)
cumulative = sorted_p.cumsum(-1)

print('ентропія: середня', round(float(entropy_bits.mean()), 2), 'біта',
      '· межа для словника', round(math.log2(V), 2))
print('у найкращого слова в середньому', round(float(sorted_p[:, 0].mean()), 4), 'маси')
print()
print('%-8s %10s %10s %10s %10s' % ('маса p', 'медіана', 'дев. дес.', 'максимум', 'середнє'))
NEED = {}
for p in (0.5, 0.8, 0.9, 0.95, 0.99):
    k = (cumulative < p).sum(-1) + 1          # +1: беремо ще одне, щоб перевищити поріг
    NEED[p] = (int(k.median()), int(k.float().quantile(0.9)), int(k.max()))
    print('%-8.2f %10d %10d %10d %10.1f' % (p, NEED[p][0], NEED[p][1], NEED[p][2],
                                            float(k.float().mean())))
print()
for k in (1, 5, 40, 100):
    print('top-k =', k, '— забирає в середньому', round(float(sorted_p[:, :k].sum(-1).mean()), 4), 'маси')

Дві речі варто прочитати з цієї таблиці.

**Перша.** Медіана й максимум розходяться в рази. Це і є відповідь на питання
«чому не просто `top-k`»: одне стале число кандидатів або ріже розумні варіанти
там, де модель вагається, або тягне зайві там, де вона певна.

**Друга.** Подивись на масу, яку забирають 40 найкращих. Те, що лишилось, — це
**хвіст**: тисячі слів, кожне з мізерною ймовірністю, а разом помітні. Саме його
й прибирають `top-k` і `top-p`.

## 7 · Пишемо декодери

Усі способи, крім променя, вміщуються в одну функцію. Логіка однакова: беремо
логіти (сирі оцінки моделі), **міняємо їх** за обраним правилом, обираємо токен,
дописуємо, повторюємо.

- **жадібне** — беремо `argmax`, тобто найбільший логіт;
- **температура** — ділимо всі логіти на `T` перед `softmax`: менша за одиницю
  робить розподіл гострішим, більша — пласкішим;
- **`top-k`** — лишаємо `k` найбільших логітів, решті ставимо мінус
  нескінченність (після `softmax` вони дадуть нуль);
- **`top-p`** — сортуємо, беремо стільки найкращих, щоб сума вперше перевищила
  `p`, решту прибираємо.

Крім того, функція вміє три речі, яких у лекційних таблицях немає, і саме вони
знадобляться в другій половині зошита:

- **`ban_eos`** і **`ban_unk`** — заборонити окремий токен: кінець рядка (це дає
  **примусову довжину**) або службовий `<unk>`;
- **`rep_pen`** — штраф за повтор: логіт слова, яке вже було, ділимо на `θ`
  (а якщо він відʼємний — множимо, інакше вийшло б навпаки, нагорода);
- **`no_rep`** — заборона повторити n-граму, яка у виході вже трапилась.

Повертаємо не лише токени, а й **суму логарифмів їхніх імовірностей за
незміненим розподілом** — це те, що промінь максимізує, і нам треба буде його
порівнювати.

In [ ]:
def ngrams(seq, n):
    return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]


@torch.no_grad()
def generate(mode, param=None, seed=0, ban_eos=False, ban_unk=False,
             rep_pen=1.0, no_rep=0):
    """Повертає [(токени без <eos>, сума логарифмів їхніх імовірностей)]."""
    gen = torch.Generator().manual_seed(seed)
    seqs = [list(p) for p in prompts]
    done = [False] * len(seqs)
    out = [[] for _ in seqs]
    logp_sum = [0.0] * len(seqs)

    for _ in range(MAXNEW):
        if all(done):
            break
        x = pad_batch(seqs)
        logits = model(x)
        last = torch.stack([logits[i, len(seqs[i]) - 1] for i in range(len(seqs))])
        # чесні логарифми ймовірностей — рахуємо ДО будь-яких правок
        true_logp = F.log_softmax(last, dim=-1)

        z = last.clone()
        # найпростіше обмежене декодування: просто забороняємо токен
        if ban_eos:
            z[:, EOS] = -float('inf')
        if ban_unk:
            z[:, UNK] = -float('inf')
        if rep_pen != 1.0:
            for i in range(len(seqs)):
                for t in set(out[i]):
                    z[i, t] = z[i, t] / rep_pen if z[i, t] > 0 else z[i, t] * rep_pen
        if no_rep:
            for i in range(len(seqs)):
                tail = tuple(out[i][-(no_rep - 1):]) if no_rep > 1 else ()
                for gram in ngrams(out[i], no_rep):
                    if gram[:-1] == tail:
                        z[i, gram[-1]] = -float('inf')

        if mode == 'greedy':
            nxt = z.argmax(-1)
        else:
            if mode == 'temp':
                z = z / param
            elif mode == 'topk':
                kth = z.topk(param, dim=-1).values[:, -1:]
                z = z.masked_fill(z < kth, -float('inf'))
            elif mode == 'topp':
                srt, idx = z.sort(-1, descending=True)
                pr = F.softmax(srt, dim=-1)
                # cumsum − pr = маса ДО цього слова; так перше слово, що перетнуло
                # поріг, лишається в грі, а все після нього викидається
                srt = srt.masked_fill((pr.cumsum(-1) - pr) > param, -float('inf'))
                z = torch.full_like(z, -float('inf')).scatter(1, idx, srt)
            nxt = torch.multinomial(F.softmax(z, dim=-1), 1, generator=gen).squeeze(1)

        for i in range(len(seqs)):
            if done[i]:
                continue
            t = int(nxt[i])
            seqs[i].append(t)
            if t == EOS:
                done[i] = True
            else:
                out[i].append(t)
                logp_sum[i] += float(true_logp[i, t])
    return list(zip(out, logp_sum))


print('готово: одна функція на пʼять правил відбору й три обмеження')
print('правила   : greedy · temp · topk · topp')
print('обмеження : ban_eos (примусова довжина) · rep_pen (штраф) · no_rep (заборона n-грам)')

### Перевірка: наш `softmax` із температурою — той самий, що в бібліотеці

Найдешевший спосіб не мати мовчазної помилки — порахувати те саме двічі різними
руками. Спершу найпростіше: напишемо `softmax` із температурою за формулою й
звіримо з бібліотечним.

In [ ]:
def softmax_by_hand(logits, temperature=1.0):
    """p = exp(z / T) поділити на суму всіх exp(z / T)."""
    scaled = [z / temperature for z in logits]
    # віднімаємо максимум: exp від великого числа переповнюється,
    # а від зсуву всіх доданків на сталу частка не міняється
    top = max(scaled)
    weights = [math.exp(z - top) for z in scaled]
    total = sum(weights)
    return [w / total for w in weights]


sample_logits = model(pad_batch(prompts[:1]))[0, len(prompts[0]) - 1]
for T in (0.5, 1.0, 2.0):
    ours = torch.tensor(softmax_by_hand(sample_logits.tolist(), T))
    theirs = F.softmax(sample_logits / T, dim=-1)
    assert torch.allclose(ours, theirs, atol=1e-6), ('розійшлось на T=' + str(T))
    print('T =', T, '· найбільша ймовірність', round(float(theirs.max()), 5),
          '· збігається з бібліотечною')
print('✅ у softmax немає магії: це рівно та формула, що в лекції')

### Перевірка: наш `top-p` робить те, що ми думаємо

Те саме зробимо з відбором кандидатів: візьмемо один розподіл, застосуємо наш
векторний `top-p` і той самий відбір найпростішим циклом, і звіримо **множини**
кандидатів.

In [ ]:
def topp_by_hand(probs, p):
    """Найпростіший, найповільніший, найзрозуміліший top-p."""
    order = sorted(range(len(probs)), key=lambda i: -probs[i])
    kept, mass = [], 0.0
    for i in order:
        kept.append(i)
        mass += probs[i]
        if mass > p:                 # перше слово, що перетнуло поріг, лишається
            break
    return set(kept)


row = P1[0].tolist()
for p in (0.5, 0.9, 0.95):
    by_hand = topp_by_hand(row, p)
    srt, idx = P1[0:1].sort(-1, descending=True)
    keep = (srt.cumsum(-1) - srt) <= p
    vectorised = {int(idx[0, j]) for j in range(V) if bool(keep[0, j])}
    assert by_hand == vectorised, ('розійшлось на p=' + str(p))
    print('p =', p, '— кандидатів', len(by_hand), '· збігається з ручним відбором')
print('✅ наш top-p відбирає рівно те саме, що й перебір у циклі')

## 8 · Промінь власними руками

`beam search` тримає одночасно `width` найкращих незавершених версій. На кожному
кроці кожну версію продовжуємо `width` найкращими токенами, з усіх кандидатів
лишаємо знову `width` найкращих — і так далі.

Дві деталі, які варто побачити в коді.

**Перша: версія, що обрала `<eos>`, більше не росте**, але й далі бере участь у
відборі. Через це готові короткі версії поступово витісняють живі довгі: до
відʼємної суми живої версії щокроку додається ще один відʼємний доданок.

**Друга: оцінок дві.** Пошукова — це те, за чим відбирають (сума логарифмів,
за бажанням поділена на довжину в степені `alpha`). Звітна — сума логарифмів
**без** `<eos>`: тільки так число буде порівнянним із тим, що повертає
`generate`, де `<eos>` теж не рахується.

In [ ]:
@torch.no_grad()
def beam_search(width, alpha=0.0, ban_eos=False, npr=NBEAM):
    """alpha=0 — проста сума логарифмів; alpha=1 — середній логарифм на токен."""
    result = []
    for prompt in prompts[:npr]:
        start = len(prompt)

        def score(b):
            length = max(len(b[0]) - start, 1)
            return b[1] / length ** alpha if alpha else b[1]

        # версія: (токени, оцінка пошуку, оцінка без <eos>, чи готова)
        beams = [(list(prompt), 0.0, 0.0, False)]
        for _ in range(MAXNEW):
            alive = [b for b in beams if not b[3]]
            if not alive:
                break
            logp = F.log_softmax(model(pad_batch([b[0] for b in alive])), dim=-1)
            if ban_eos:
                logp[:, :, EOS] = -float('inf')
            candidates = [b for b in beams if b[3]]        # готові переносимо як є
            for bi, b in enumerate(alive):
                values, ids = logp[bi, len(b[0]) - 1].topk(width)
                for k in range(width):
                    t, dv = int(ids[k]), float(values[k])
                    candidates.append((b[0] + [t], b[1] + dv,
                                       b[2] + (0.0 if t == EOS else dv), t == EOS))
            beams = sorted(candidates, key=score, reverse=True)[:width]
        best = max(beams, key=score)
        result.append(([t for t in best[0][start:] if t != EOS], best[2]))
    return result


print('готово: промінь із двома оцінками — пошуковою і звітною')
print('перевіримо його наступною клітинкою, і перевірка буде сувора')

### Перевірка: промінь ширини 1 — це рівно жадібне декодування

Ширина 1 не дає променю жодного вибору: він щокроку бере найкращий токен і
нічого не тримає про запас. Отже його вихід мусить **посимвольно** збігтися з
жадібним. Якщо ні — десь помилка.

In [ ]:
greedy_out = generate('greedy')
beam1_out = beam_search(1, npr=20)
for i in range(20):
    assert greedy_out[i][0] == beam1_out[i][0], 'промінь ширини 1 розійшовся з жадібним на ' + str(i)
print('✅ промінь ширини 1 дає рівно те саме, що жадібне декодування')
print()
print('приклад:', show(greedy_out[0][0]) or '(порожньо)')

## 9 · Чим міряти вихід

Пʼять величин, і жодна з них поодинці не є якістю.

| величина | що каже | чим бреше |
|---|---|---|
| **довжина** | скільки слів вийшло | сама собою нічого не значить |
| **цикл** | частка виходів, де чотири слова поспіль повторились | не бачить безбарвності |
| **distinct-2** | частка різних пар слів серед породжених | випадковий шум дає рівно 1.0 |
| **F1** | наскільки набір слів збігся з тим, що стояло насправді | у відкритій задачі еталона немає |
| **logp/токен** | середній логарифм імовірності | міряє згоду моделі з собою, а не якість |

`F1` тут — це звичайна гармонійна середня точності й повноти по **мультимножині
слів**: скільки слів ми вгадали, поділити на те, скільки сказали і скільки треба
було.

In [ ]:
def measure(gens):
    """Пʼять чисел на набір продовжень."""
    lengths = [len(g) for g, _ in gens]
    loops = sum(1 for g, _ in gens
                if len(ngrams(g, 4)) > len(set(ngrams(g, 4))))
    dist2 = [len(set(ngrams(g, 2))) / len(ngrams(g, 2)) for g, _ in gens if len(g) > 1]
    f1 = []
    for (g, _), ref in zip(gens, refs[:len(gens)]):
        said, need = collections.Counter(g), collections.Counter(ref)
        overlap = sum((said & need).values())
        f1.append(0.0 if overlap == 0 else 2 * overlap / (len(g) + len(ref)))
    per_token = [lp / len(g) for g, lp in gens if len(g) > 0]
    return dict(length=round(statistics.mean(lengths), 2),
                loop=round(loops / len(gens), 4),
                distinct2=round(statistics.mean(dist2), 4) if dist2 else 0.0,
                f1=round(statistics.mean(f1), 4),
                logp=round(statistics.mean(per_token), 4) if per_token else 0.0)


HEAD = '%-24s %8s %8s %11s %8s %11s'
LINE = '%-24s %8.2f %8.4f %11.4f %8.4f %11.4f'
def row(name, s):
    print(LINE % (name, s['length'], s['loop'], s['distinct2'], s['f1'], s['logp']))

print(HEAD % ('спосіб', 'довжина', 'цикл', 'distinct-2', 'F1', 'logp/токен'))
print('-' * 74)
row('еталон (людина)', dict(length=statistics.mean(len(r) for r in refs), loop=0,
                            distinct2=statistics.mean(len(set(ngrams(r, 2))) / len(ngrams(r, 2))
                                                      for r in refs if len(r) > 1),
                            f1=1.0, logp=0.0))

## 10 · Тринадцять способів на одній моделі

Тепер прогін. Модель одна й та сама, підказки ті самі — міняється лише правило
вибору.

In [ ]:
WAYS = []
t0 = time.process_time()
WAYS.append(('жадібне', greedy_out))
for T in (0.7, 1.0, 1.3):
    WAYS.append(('температура ' + str(T), generate('temp', T, seed=1)))
for k in (5, 40):
    WAYS.append(('top-k ' + str(k), generate('topk', k, seed=1)))
for p in (0.9, 0.95):
    WAYS.append(('top-p ' + str(p), generate('topp', p, seed=1)))
print('семплювання:', round(time.process_time() - t0, 1), 'с')

t0 = time.process_time()
BEAM = {}
for width in (2, 4, 8):
    for alpha in (0.0, 1.0):
        name = 'промінь ' + str(width) + (' + норм.' if alpha else '')
        BEAM[name] = beam_search(width, alpha)
        WAYS.append((name, BEAM[name]))
print('промінь:', round(time.process_time() - t0, 1), 'с')

STATS = [(name, measure(gens)) for name, gens in WAYS]
print()
print(HEAD % ('спосіб', 'довжина', 'цикл', 'distinct-2', 'F1', 'logp/токен'))
print('-' * 74)
for name, s in STATS:
    row(name, s)

Головне тут — **два останні стовпчики**. `logp/токен` — це те, що промінь
максимізує; `F1` — те, заради чого все робилось. Подивись, чи йдуть вони в один
бік.

Порахуймо це числом: кореляцію між ними по всіх тринадцятьох способах.

In [ ]:
xs = [s['logp'] for _, s in STATS]
ys = [s['f1'] for _, s in STATS]
mx, my = statistics.mean(xs), statistics.mean(ys)
num = sum((a - mx) * (b - my) for a, b in zip(xs, ys))
den = math.sqrt(sum((a - mx) ** 2 for a in xs) * sum((b - my) ** 2 for b in ys))
CORR = num / den

best_logp = max(STATS, key=lambda t: t[1]['logp'])
best_f1 = max(STATS, key=lambda t: t[1]['f1'])
print('кореляція logp/токен із F1:', round(CORR, 4))
print()
print('найправдоподібніший спосіб:', best_logp[0], '— F1', best_logp[1]['f1'])
print('найточніший спосіб        :', best_f1[0], '— logp/токен', best_f1[1]['logp'])

## 11 · Примусова довжина: тут і зʼявляється виродження

На природній довжині циклів майже немає — і це не тому, що модель хороша, а
тому, що **наші рядки короткі**. Після трьох слів підказки модель часто вважає
найімовірнішим просто кінець рядка, обриває продовження й до циклу не доходить.

Щоб побачити виродження, треба змусити модель говорити. Заборонимо `<eos>` і
візьмемо рівно двадцять токенів у всіх. Це найпростіший різновид **обмеженого
декодування**: ми не міняємо модель, а лише забороняємо їй один токен.

In [ ]:
FORCED = [('жадібне', generate('greedy', ban_eos=True))]
for T in (0.7, 1.0):
    FORCED.append(('температура ' + str(T), generate('temp', T, seed=1, ban_eos=True)))
for k in (5, 40):
    FORCED.append(('top-k ' + str(k), generate('topk', k, seed=1, ban_eos=True)))
for p in (0.9, 0.95):
    FORCED.append(('top-p ' + str(p), generate('topp', p, seed=1, ban_eos=True)))
for width in (4, 8):
    FORCED.append(('промінь ' + str(width), beam_search(width, ban_eos=True)))

FORCED_STATS = [(name, measure(gens)) for name, gens in FORCED]
print(HEAD % ('спосіб, довжина 20', 'довжина', 'цикл', 'distinct-2', 'F1', 'logp/токен'))
print('-' * 74)
for name, s in FORCED_STATS:
    row(name, s)

In [ ]:
# подивимось очима: те саме речення, дописане різними способами
for name, gens in FORCED:
    if name in ('жадібне', 'температура 1.0', 'top-p 0.9', 'промінь 8'):
        print('%-18s %s' % (name + ':', show(gens[0][0])))
print('%-18s %s' % ('а насправді:', show(refs[0])))
print()
print('підказка була:', show(prompts[0][1:]))

### Перше, що робить справжній декодер: забороняє службові токени

Подивись на вихід жадібного декодування уважно. Якщо він складається з
`<unk>` — це не збіг. `<unk>` заміняє всі рідкісні слова корпусу, тому в
навчальних даних він трапляється частіше за будь-яке справжнє слово, і
жадібне правило слухняно його бере. Далі контекст майже не змінюється, найкращим
кандидатом знову виявляється `<unk>`, і так до кінця: це **нерухома точка**
жадібного вибору.

Жоден робочий декодер `<unk>` не породжує — його забороняють першим рядком, і це
рівно та сама операція, що й примусова довжина: логіту ставлять мінус
нескінченність. Подивімось, що зміниться.

In [ ]:
no_unk = generate('greedy', ban_eos=True, ban_unk=True)
print(HEAD % ('жадібне, довжина 20', 'довжина', 'цикл', 'distinct-2', 'F1', 'logp/токен'))
print('-' * 74)
row('як є', measure(dict(FORCED)['жадібне']))
row('<unk> заборонено', measure(no_unk))
print()
for i in range(3):
    print('підказка :', show(prompts[i][1:]))
    print('вихід    :', show(no_unk[i][0]))
    print('еталон   :', show(refs[i]))
    print()

## 12 · Чи справді повтор себе підсилює

У великих моделей описано другий механізм циклу, тонший за нерухому точку. У
справжніх текстах повтори бувають — списки, шаблонні повідомлення, приспіви, —
тож модель могла вивчити: **текст, у якому фраза вже трапилась, і далі частіше
містить її ж**. Тоді кожен новий повтор ставав би ще одним свідченням на користь
наступного, і ймовірність повтору росла б із кожним разом.

Це **гіпотеза**, і її можна перевірити прямо. Візьмемо три найчастіші пари слів
навчального корпусу, допишемо кожну до справжніх підказок `k` разів поспіль і
подивимось, з якою ймовірністю модель почне її вкотре.

⚠️ Одна пастка, у яку легко потрапити: якщо контекст складається **лише** з
повторів фрази, то `k = 0` і `k = 1` — це два зовсім різні контексти, і різниця
між ними нічого не каже про підсилення. Тому повтори дописуємо до **справжніх**
підказок, а порівнюємо точки від `k = 1` і далі.

In [ ]:
bigrams = collections.Counter()
for s in train:
    body = [t for t in s if t not in (BOS, EOS)]
    for i in range(len(body) - 1):
        bigrams[(body[i], body[i + 1])] += 1
# службові токени з фраз викидаємо: повтор <unk> — це інше явище
top_pairs = [p for p, c in bigrams.most_common(40) if UNK not in p][:3]


@torch.no_grad()
def repeat_curve(phrase, k_max=8):
    """Ймовірність почати фразу знову, коли вона вже стоїть у контексті k разів.

    Важливо: повтори дописуємо до СПРАВЖНІХ підказок і усереднюємо по всіх.
    Якщо контекст складається лише з повторів, то k=0 і k=1 — це два різні
    контексти, і порівнювати їх немає сенсу."""
    first = phrase[0]
    out = []
    for k in range(k_max):
        contexts = [(list(p) + list(phrase) * k)[:MAXLEN] for p in prompts]
        logits = model(pad_batch(contexts))
        last = torch.stack([logits[i, len(contexts[i]) - 1] for i in range(len(contexts))])
        out.append(round(float(F.softmax(last, dim=-1)[:, first].mean()), 4))
    return out


LOOP_CURVES = []
for pair in top_pairs:
    curve = repeat_curve(pair)
    LOOP_CURVES.append((itos[pair[0]] + ' ' + itos[pair[1]], curve))
    print('%-24s %s' % (itos[pair[0]] + ' ' + itos[pair[1]], curve))
print()
print('порівнюй точки від 1 і далі: точка 0 — це інший контекст, без фрази взагалі')
for name, curve in LOOP_CURVES:
    grew = curve[-1] / max(curve[1], 1e-9)
    print('%-24s після сімох повторів у %.2f раза %s, ніж після одного'
          % (name, grew if grew >= 1 else 1 / grew, 'більше' if grew >= 1 else 'МЕНШЕ'))

Ось чому цикл частіший саме в жадібного: воно **детерміноване**. Щойно
ймовірність повтору стала найбільшою, жадібне правило зобовʼязане її взяти, а
контекст після цього має рівно той самий вигляд, що й до. Вийти нема як.
Семплювання в тій самій ситуації має шанс `1 − p` відхилитись — щоправда, чим
вища `p`, тим менший той шанс.

## 13 · Два лікування й ціна кожного

**Штраф за повтор.** Логіт слова, яке вже було у виході, ділимо на `θ ≥ 1`.
Якщо логіт відʼємний — множимо: інакше ділення наблизило б його до нуля, тобто
**нагородило** б слово замість покарати.

**Заборона повторених n-грам.** Жорсткіше: якщо крок утворить n-граму, яка у
виході вже була, цей крок просто заборонено.

Обидва прийоми живуть **поза** моделлю: вони міняють логіти перед вибором.
І обидва мають однакову природну ваду — вони не вміють відрізнити поганий
повтор від законного. Дивимось, що виграємо і що втрачаємо.

In [ ]:
print(HEAD % ('штраф за повтор', 'довжина', 'цикл', 'distinct-2', 'F1', 'logp/токен'))
print('-' * 74)
PEN = []
for theta in (1.0, 1.05, 1.1, 1.2, 1.5, 2.0):
    s = measure(generate('greedy', ban_eos=True, rep_pen=theta))
    PEN.append((theta, s))
    row('θ = ' + str(theta), s)

In [ ]:
print(HEAD % ('заборона n-грам', 'довжина', 'цикл', 'distinct-2', 'F1', 'logp/токен'))
print('-' * 74)
base = measure(generate('greedy', ban_eos=True))
row('без заборони', base)
for n in (2, 3, 4):
    row('заборонено ' + str(n) + '-грами', measure(generate('greedy', ban_eos=True, no_rep=n)))

Заборона біграм (`n = 2`) прибирає цикли повністю — бо вона забороняє **будь-яку
пару слів двічі**. Але це вже не лікування, а каліцтво: у мові пари слів
законно повторюються, і метрика збігу з еталоном це показує.

## 14 · Скільки помилок пошуку робить жадібне декодування

Останній замір. Жадібне правило бере найкраще слово на кожному кроці — але це
не те саме, що найкраща **послідовність**. Порахуємо, як часто промінь ширини 8
знаходить продовження з **вищою** сумою логарифмів, ніж жадібне.

І одразу поставимо друге питання, важливіше за перше: а чи стало це продовження
**кращим**?

In [ ]:
greedy_cut = greedy_out[:NBEAM]
beam8 = BEAM['промінь 8']
better = sum(1 for (g, lg), (b, lb) in zip(greedy_cut, beam8) if lb > lg + 1e-6)
same = sum(1 for (g, _), (b, _) in zip(greedy_cut, beam8) if g == b)

print('підказок у порівнянні            ', len(greedy_cut))
print('промінь знайшов імовірніше       ', better, '=', round(better / len(greedy_cut), 4))
print('вихід збігся з жадібним          ', same, '=', round(same / len(greedy_cut), 4))
print()
print('F1 жадібного   ', measure(greedy_cut)['f1'])
print('F1 променя 8   ', measure(beam8)['f1'])
print('logp/токен жадібного', measure(greedy_cut)['logp'])
print('logp/токен променя 8', measure(beam8)['logp'])

## Що ми побачили

1. **Правдоподібність і якість тягнуть у різні боки.** Ширший промінь знаходить
   імовірніші продовження — і саме тому вкорочує їх і робить безбарвними.
2. **Жадібне декодування — не найімовірніша послідовність.** Промінь регулярно
   знаходить кращу за його власною міркою; від цього вихід не стає кращим для
   людини.
3. **Хвіст розподілу треба різати, і `top-p` робить це розумніше за `top-k`**,
   бо кількість кандидатів на різних кроках відрізняється в рази.
4. **Виродження — не поломка, а нерухома точка.** Жадібне правило
   детерміноване: щойно найкращим кандидатом став токен, після якого контекст
   майже не змінюється, вибір повторюватиметься вічно. Перевір, чи підтвердився
   на твоїх числах другий, тонший механізм — самопідсилення повтору.
5. **Лікування коштує.** Штраф за повтор і заборона n-грам прибирають цикли,
   але платять збігом із тим, що людина написала насправді.

## Завдання трьох рівнів

**🟢 Рівень 1.** Додай до таблиці декодувань `top-p` = 0.99 і температуру 0.5.
Де вони стануть у стовпчиках `logp/токен` і `distinct-2`? Поясни словами, чому
саме там.

**🟡 Рівень 2.** Промінь у нас порівнює версії або сумою логарифмів, або
середнім на токен (`alpha` = 0 і 1). Прогони проміжні `alpha` = 0.2, 0.4, 0.6,
0.8 при ширині 8 і побудуй залежність довжини виходу від `alpha`. Знайди те
`alpha`, при якому середня довжина найближча до еталонної, і скажи, чи збігається
воно з тим, що дає найкращий `F1`.

**🔴 Рівень 3.** Напиши **типове** декодування (typical sampling): замість
найімовірніших слів лишай ті, чия несподіванка `−log p` найближча до ентропії
розподілу, доки не набереш масу `p`. Заміряй його тими самими пʼятьма
величинами й порівняй із `top-p` при однаковій масі. Гіпотеза, яку треба
перевірити: типове декодування має дати вищий `distinct-2` при тому самому рівні
циклів.

In [ ]:
print('процесорного часу на весь зошит:', round(time.process_time() - START_CPU, 1), 'с')
print('(за годинником буде більше, якщо машина зайнята — це нормально)')